In [ ]:
import sys
sys.path.insert(0, "../") # replace with path/to/project/root
from data import get_data, get_site_ids

import pandas as pd
import numpy as np


In [ ]:
uid = "WQS0083"

data = get_data(site_uid=uid)

daily_avg_precip = ( # calculate average precipitation over entire basin for each date
    data.rain.groupby("date", as_index=False)["precip_in_1d"]
      .mean()
)

# Take sum of crops over the entire basin (figure out how to normalized)
crops = data.crops.groupby("year", as_index=False).sum() 

# Calculate mean surplus nitrogen over basin in 2017. This does not work as a feature because it is constant.
nitrogen_2017 = data.surplus[data.surplus['year'] == 2017]
SURPLUS_N_2017 = nitrogen_2017['surplus_kgha'].mean()



In [ ]:
from data.water import aggregate_by_interval

# Aggregate water levels to each calendar day. I used max so we coult track violations,
# but might need to be average for a continuous model. This could explain why linreg is so bad.
water_max = aggregate_by_interval(site_uid=uid, value_col="nitrate_con", interval="1D", agg_func="max").to_frame()
water_max["date"] = water_max.index.date
water_max = water_max.reset_index(drop=True).rename(columns={'nitrate_con':'nitrate_max'})

water = aggregate_by_interval(site_uid=uid, value_col="nitrate_con", interval="1D", agg_func="mean").to_frame()
water["date"] = water.index.date
water = water.reset_index(drop=True)
water = pd.merge(water, water_max, on="date")

water

# Create true/false column if there was a single violation in a calendar day
water["violation"] = (water.nitrate_max > 10).astype(int)
print("Violation counts:\n", water["violation"].value_counts()) # Check counts of violations

water["date"] = pd.to_datetime(water["date"])
daily_avg_precip["date"] = pd.to_datetime(daily_avg_precip["date"])

df = water[["date", "violation", "nitrate_con"]].merge(
    daily_avg_precip,
    on="date",
    how="left"
)

# Add in gridded rain_x_surplus data
rain = pd.merge(data.rain, data.grid[['node_id', 'cell_area', 'geometry']], how='left', on='node_id')
rain = pd.merge(rain, nitrogen_2017, on='node_id')
rain['rain_x_surplus'] = rain.precip_in_1d * rain.surplus_kgha

daily_rain_x_surplus = ( # calculate average rain_x_surplus over entire basin for each date
    rain.groupby("date", as_index=False)["rain_x_surplus"]
      .mean()
)

df = pd.merge(df, daily_rain_x_surplus, on = 'date')

# Add in 7,14, and 30 day rolling averages
df["rain_x_surplus_7d"]  = df["rain_x_surplus"].rolling(7,  min_periods=1).sum()
df["rain_x_surplus_14d"] = df["rain_x_surplus"].rolling(14, min_periods=1).sum()
df["rain_x_surplus_30d"] = df["rain_x_surplus"].rolling(30, min_periods=1).sum()

# Add in 7,14, and 30 day rolling averages
df["rain_7d"]  = df["precip_in_1d"].rolling(7,  min_periods=1).sum()
df["rain_14d"] = df["precip_in_1d"].rolling(14, min_periods=1).sum()
df["rain_30d"] = df["precip_in_1d"].rolling(30, min_periods=1).sum()

# Crop data is yearly -> broadcast onto every day in that crop year.
df["year"] = df["date"].dt.year
df = df.merge(crops.drop(columns='node_id'), on="year", how="left")      
 
# Surplus N is a single static estimate -> constant column. Will only matter for multiple sites.
df["surplus_n_2017"] = SURPLUS_N_2017
 
# Autoregressive features: nitrate is autocorrelated, recent readings matter
df["nitrate_lag1"] = df["nitrate_con"]            # today's reading
df["nitrate_lag2"] = df["nitrate_con"].shift(1)
df["nitrate_lag3"] = df["nitrate_con"].shift(2)

df["nitrate_tomorrow"]   = df["nitrate_con"].shift(-1)
df["violation_tomorrow"] = (df["violation"].shift(-1))





In [ ]:
count = {2018: 0, 2019: 0, 2020: 0, 2021:0, 2022:0, 2023: 0, 2024:0, 2025:0, 2026:0}
for i in range(0, len(water_max)):
    date = water_max.date[i]
    count[date.year] += 1
count

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.figure(figsize = (20, 5))
sns.scatterplot(water_max, x = 'date', y = 'nitrate_max')
plt.show()


In [ ]:
from statsmodels.tsa.api import ExponentialSmoothing

# getting rid of 2018 data

water = water_max[165:]
water = water.set_index('date')
water = water.dropna(subset=['nitrate_max']) # if not already
water = water.reindex(pd.date_range(water.index.min(), water.index.max(), freq='D'))
water['nitrate_max'] = water['nitrate_max'].interpolate(method='time')


# splitting at 2025
split = 365*6
train_water_max = water.nitrate_max[:split]
test_water_max = water.nitrate_max[split:]






## First Attempt at Exponential Smoothing

In [ ]:
import matplotlib.dates as mdates
water_monthly = water['nitrate_max'].resample('ME').mean()

split = int(len(water_monthly) * 0.8)
train = water_monthly[:split]
test = water_monthly[split:]

model_hw = ExponentialSmoothing(train, trend='add', damped_trend=True, seasonal='add', seasonal_periods=12)
forecast_hw = model_hw.fit().forecast(len(test))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test.index, test.values, label='actual')
ax.plot(test.index, forecast_hw.values, label='forecast')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.title('Holt-Winters Forecast')
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
from sklearn.metrics import mean_squared_error

hw_mse = mean_squared_error(test, forecast_hw)

print(f'Holt-Winters MSE: {hw_mse:.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train.index, train.values, label='train')
ax.plot(test.index, test.values, label='actual')
ax.plot(test.index, forecast_hw.values, label='forecast')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.title('Holt-Winters Forecast')
plt.legend()
plt.tight_layout()
plt.show()

## Monthly vs Weekly Exponential Smoothing

In [ ]:
water_weekly = water['nitrate_max'].resample('W').mean()
water_weekly = water_weekly.interpolate(method='time')  # fill any remaining NaN weeks

split = int(len(water_weekly) * 0.8)
train_w = water_weekly[:split]
test_w = water_weekly[split:]

model_hw_w = ExponentialSmoothing(train_w, trend='add', damped_trend=True, seasonal='add', seasonal_periods=52)
forecast_hw_w = model_hw_w.fit().forecast(len(test_w))

mse_w = mean_squared_error(test_w, forecast_hw_w)
print(f'Weekly Holt-Winters MSE: {mse_w:.4f}')
print(f'Monthly Holt-Winters MSE: 4.9781')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Monthly
axes[0].plot(train.index, train.values, label='train')
axes[0].plot(test.index, test.values, label='actual')
axes[0].plot(test.index, forecast_hw.values, label='forecast')
axes[0].xaxis.set_major_locator(mdates.YearLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title(f'Monthly Holt-Winters (MSE: 4.9781)')
axes[0].legend()

# Weekly
axes[1].plot(train_w.index, train_w.values, label='train')
axes[1].plot(test_w.index, test_w.values, label='actual')
axes[1].plot(test_w.index, forecast_hw_w.values, label='forecast')
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title(f'Weekly Holt-Winters (MSE: {mse_w:.4f})')
axes[1].legend()

plt.tight_layout()
plt.show()

## Trying new model prophet

In [ ]:
#trying new model prophet
from prophet import Prophet

df = water_monthly.reset_index()
df.columns = ['ds', 'y']

split = int(len(df) * 0.8)
train_p = df[:split]
test_p = df[split:]

model_p = Prophet(yearly_seasonality=True)
model_p.fit(train_p)

forecast_p = model_p.predict(test_p[['ds']])

mse_p = mean_squared_error(test_p['y'], forecast_p['yhat'])
print(f'Prophet MSE: {mse_p:.4f}')
print(f'Holt-Winters MSE: 4.9781')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_p['ds'], test_p['y'].values, label='actual')
ax.plot(test_p['ds'], forecast_p['yhat'].values, label='forecast')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.title('Prophet Forecast')
plt.legend()
plt.tight_layout()
plt.show()

## Prophet with regressors. 

In [ ]:
# Resample features to monthly
feature_cols = ['precip_in_1d', 'rain_x_surplus_7d', 'rain_x_surplus_14d', 'rain_x_surplus_30d']



df['date'] = pd.to_datetime(df['date'])
df_monthly = df.set_index('date')[['nitrate_con'] + feature_cols].resample('ME').mean().reset_index()
df_monthly = df_monthly.dropna()  # drop any months with NaN in any column
df_monthly.columns = ['ds', 'y'] + feature_cols

split = int(len(df_monthly) * 0.8)
train_p = df_monthly[:split]
test_p  = df_monthly[split:]

model_p = Prophet(yearly_seasonality=True)
for col in feature_cols:
    model_p.add_regressor(col)

model_p.fit(train_p)

forecast_p = model_p.predict(test_p[['ds'] + feature_cols])

mse_p = mean_squared_error(test_p['y'], forecast_p['yhat'])
print(f'Prophet with regressors MSE: {mse_p:.4f}')
print(f'Holt-Winters MSE:            4.9781')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_p['ds'], test_p['y'].values, label='actual')
ax.plot(test_p['ds'], forecast_p['yhat'].values, label='forecast')
ax.fill_between(test_p['ds'], forecast_p['yhat_lower'].values, forecast_p['yhat_upper'].values, alpha=0.2, label='uncertainty')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.title('Prophet Forecast with Regressors')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = ['precip_in_1d', 'rain_x_surplus_7d', 'rain_x_surplus_14d', 'rain_x_surplus_30d']


df_weekly = df.set_index('date')[['nitrate_con'] + feature_cols].resample('W').mean().reset_index()
df_weekly = df_weekly.dropna()
df_weekly.columns = ['ds', 'y'] + feature_cols

split_w = int(len(df_weekly) * 0.8)
train_pw = df_weekly[:split_w]
test_pw  = df_weekly[split_w:]

model_pw = Prophet(yearly_seasonality=True, weekly_seasonality=False)
for col in feature_cols:
    model_pw.add_regressor(col)

model_pw.fit(train_pw)

forecast_pw = model_pw.predict(test_pw[['ds'] + feature_cols])

mse_pw = mean_squared_error(test_pw['y'], forecast_pw['yhat'])
print(f'Prophet Weekly with regressors MSE: {mse_pw:.4f}')
print(f'Prophet Monthly with regressors MSE: 5.7714')
print(f'Holt-Winters Monthly MSE:            4.9781')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_pw['ds'], test_pw['y'].values, label='actual')
ax.plot(test_pw['ds'], forecast_pw['yhat'].values, label='forecast')
ax.fill_between(test_pw['ds'], forecast_pw['yhat_lower'].values, forecast_pw['yhat_upper'].values, alpha=0.2, label='uncertainty')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.title('Prophet Weekly Forecast with Regressors')
plt.legend()
plt.tight_layout()
plt.show()

## Prophet and Exponential Smoothing Cross Validation

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import numpy as np

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

hw_mses = []
prophet_mses = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(df_monthly)):

    train_fold = df_monthly.iloc[train_idx].set_index('ds')
    test_fold  = df_monthly.iloc[test_idx].set_index('ds')

    if len(train_fold) < 24:
        print(f"Fold {fold+1}: skipped (only {len(train_fold)} months of training data)")
        continue

    # Holt-Winters
    hw = ExponentialSmoothing(
        train_fold['y'],
        trend='add', damped_trend=True,
        seasonal='add', seasonal_periods=12
    )
    hw_fit = hw.fit()
    hw_forecast = hw_fit.forecast(len(test_fold))
    hw_mses.append(mean_squared_error(test_fold['y'], hw_forecast))

    # Prophet
    train_p_fold = train_fold.reset_index()
    test_p_fold  = test_fold.reset_index()
    m = Prophet(yearly_seasonality=True)
    for col in feature_cols:
        m.add_regressor(col)
    m.fit(train_p_fold)
    fc = m.predict(test_p_fold[['ds'] + feature_cols])
    prophet_mses.append(mean_squared_error(test_p_fold['y'], fc['yhat']))

    print(f"Fold {fold+1}: HW MSE={hw_mses[-1]:.4f}  Prophet MSE={prophet_mses[-1]:.4f}")

print(f"\nHolt-Winters  MSE: {np.mean(hw_mses):.4f} +/- {np.std(hw_mses):.4f}")
print(f"Prophet       MSE: {np.mean(prophet_mses):.4f} +/- {np.std(prophet_mses):.4f}")

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import numpy as np

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

hw_mses = []
prophet_mses = []

n_valid_folds = sum(1 for train_idx, _ in tscv.split(df_monthly) if len(train_idx) >= 24)
fig, axes = plt.subplots(n_valid_folds, 2, figsize=(16, 4 * n_valid_folds))
plot_idx = 0

for fold, (train_idx, test_idx) in enumerate(tscv.split(df_monthly)):

    train_fold = df_monthly.iloc[train_idx].set_index('ds')
    test_fold  = df_monthly.iloc[test_idx].set_index('ds')

    if len(train_fold) < 24:
        print(f"Fold {fold+1}: skipped (only {len(train_fold)} months of training data)")
        continue

    # Holt-Winters
    hw = ExponentialSmoothing(
        train_fold['y'],
        trend='add', damped_trend=True,
        seasonal='add', seasonal_periods=12
    )
    hw_fit = hw.fit()
    hw_forecast = hw_fit.forecast(len(test_fold))
    hw_mses.append(mean_squared_error(test_fold['y'], hw_forecast))

    # Prophet
    train_p_fold = train_fold.reset_index()
    test_p_fold  = test_fold.reset_index()
    m = Prophet(yearly_seasonality=True)
    for col in feature_cols:
        m.add_regressor(col)
    m.fit(train_p_fold)
    fc = m.predict(test_p_fold[['ds'] + feature_cols])
    prophet_mses.append(mean_squared_error(test_p_fold['y'], fc['yhat']))

    print(f"Fold {fold+1}: HW MSE={hw_mses[-1]:.4f}  Prophet MSE={prophet_mses[-1]:.4f}")

    # Holt-Winters plot
    ax = axes[plot_idx, 0]
    ax.plot(train_fold.index, train_fold['y'].values, label='train', color='gray')
    ax.plot(test_fold.index, test_fold['y'].values, label='actual')
    ax.plot(test_fold.index, hw_forecast.values, label='forecast')
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(f'Fold {fold+1} - Holt-Winters (MSE={hw_mses[-1]:.4f})')
    ax.legend()

    # Prophet plot
    ax = axes[plot_idx, 1]
    ax.plot(train_fold.index, train_fold['y'].values, label='train', color='gray')
    ax.plot(test_p_fold['ds'], test_p_fold['y'].values, label='actual')
    ax.plot(test_p_fold['ds'], fc['yhat'].values, label='forecast')
    ax.fill_between(test_p_fold['ds'], fc['yhat_lower'].values, fc['yhat_upper'].values, alpha=0.2, label='uncertainty')
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(f'Fold {fold+1} - Prophet (MSE={prophet_mses[-1]:.4f})')
    ax.legend()

    plot_idx += 1

plt.suptitle('Cross-Validation Forecasts: Holt-Winters vs Prophet', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nHolt-Winters  MSE: {np.mean(hw_mses):.4f} +/- {np.std(hw_mses):.4f}")
print(f"Prophet       MSE: {np.mean(prophet_mses):.4f} +/- {np.std(prophet_mses):.4f}")

## Just Regular XGBoost without Feature Engineering

In [ ]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import numpy as np

# Use monthly aggregated features
feature_cols_xgb = ['nitrate_lag1', 'nitrate_lag2', 'nitrate_lag3',
                     'rain_x_surplus_7d', 'rain_x_surplus_14d', 'rain_x_surplus_30d',
                     'precip_in_1d', 'rain_7d', 'rain_14d', 'rain_30d']

# Aggregate to monthly
df_xgb = df.set_index('date')[feature_cols_xgb + ['nitrate_con']].resample('ME').mean().reset_index()
df_xgb = df_xgb.dropna()

# Add month as a feature to capture seasonality
df_xgb['month'] = df_xgb['date'].dt.month

X = df_xgb[feature_cols_xgb + ['month']].values
y = df_xgb['nitrate_con'].values

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
xgb_mses = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    if len(X_train) < 24:
        print(f"Fold {fold+1}: skipped")
        continue

    model_xgb = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
    model_xgb.fit(X_train, y_train)
    preds = model_xgb.predict(X_test)
    xgb_mses.append(mean_squared_error(y_test, preds))

    print(f"Fold {fold+1}: XGB MSE={xgb_mses[-1]:.4f}")

print(f"\nXGBoost       MSE: {np.mean(xgb_mses):.4f} +/- {np.std(xgb_mses):.4f}")
print(f"Prophet       MSE: 11.1755 +/- 6.7159")
print(f"Holt-Winters  MSE: 14.2153 +/- 7.5739")

In [ ]:
n_valid_folds = sum(1 for train_idx, _ in tscv.split(X) if len(train_idx) >= 24)
fig, axes = plt.subplots(n_valid_folds, 1, figsize=(12, 4 * n_valid_folds))
plot_idx = 0
xgb_mses = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    if len(X_train) < 24:
        print(f"Fold {fold+1}: skipped")
        continue

    model_xgb = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
    model_xgb.fit(X_train, y_train)
    preds = model_xgb.predict(X_test)
    xgb_mses.append(mean_squared_error(y_test, preds))

    dates_train = df_xgb['date'].iloc[train_idx]
    dates_test  = df_xgb['date'].iloc[test_idx]

    ax = axes[plot_idx]
    ax.plot(dates_train.values[-24:], y_train[-24:], label='train (last 24mo)', color='gray')
    ax.plot(dates_test.values, y_test, label='actual')
    ax.plot(dates_test.values, preds, label='forecast')
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(f'Fold {fold+1} - XGBoost (MSE={xgb_mses[-1]:.4f})')
    ax.legend()

    print(f"Fold {fold+1}: XGB MSE={xgb_mses[-1]:.4f}")
    plot_idx += 1

plt.suptitle('XGBoost Cross-Validation Forecasts', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nXGBoost       MSE: {np.mean(xgb_mses):.4f} +/- {np.std(xgb_mses):.4f}")
print(f"Prophet       MSE: 11.1755 +/- 6.7159")
print(f"Holt-Winters  MSE: 14.2153 +/- 7.5739")

In [ ]:
test_sites = ['WQS0039', 'WQS0054', 'WQS0081', 'WQS0050']

for site in test_sites:
    d = get_data(site_uid=site)
    w = aggregate_by_interval(site_uid=site, value_col="nitrate_con", interval="1D", agg_func="mean").to_frame()
    print(f"{site}: {len(w)} days, {w.index.min().date()} to {w.index.max().date()}")

In [ ]:
test_sites = ['WQS0039', 'WQS0054', 'WQS0081', 'WQS0050']

def build_features(site_uid):
    d = get_data(site_uid=site_uid)
    
    daily_avg_precip = d.rain.groupby("date", as_index=False)["precip_in_1d"].mean()
    nitrogen_2017 = d.surplus[d.surplus['year'] == 2017]
    rain = pd.merge(d.rain, d.grid[['node_id', 'cell_area', 'geometry']], how='left', on='node_id')
    rain = pd.merge(rain, nitrogen_2017, on='node_id')
    rain['rain_x_surplus'] = rain.precip_in_1d * rain.surplus_kgha
    daily_rain_x_surplus = rain.groupby("date", as_index=False)["rain_x_surplus"].mean()

    w = aggregate_by_interval(site_uid=site_uid, value_col="nitrate_con", interval="1D", agg_func="mean").to_frame()
    w["date"] = pd.to_datetime(w.index.date)
    w = w.reset_index(drop=True)

    df = w.merge(daily_avg_precip, on="date", how="left")
    df = pd.merge(df, daily_rain_x_surplus, on='date')

    df["rain_x_surplus_7d"]  = df["rain_x_surplus"].rolling(7,  min_periods=1).sum()
    df["rain_x_surplus_14d"] = df["rain_x_surplus"].rolling(14, min_periods=1).sum()
    df["rain_x_surplus_30d"] = df["rain_x_surplus"].rolling(30, min_periods=1).sum()
    df["rain_7d"]  = df["precip_in_1d"].rolling(7,  min_periods=1).sum()
    df["rain_14d"] = df["precip_in_1d"].rolling(14, min_periods=1).sum()
    df["rain_30d"] = df["precip_in_1d"].rolling(30, min_periods=1).sum()

    df["nitrate_lag1"] = df["nitrate_con"]
    df["nitrate_lag2"] = df["nitrate_con"].shift(1)
    df["nitrate_lag3"] = df["nitrate_con"].shift(2)
    df["month"] = df["date"].dt.month

    df_monthly = df.set_index('date')[feature_cols_xgb + ['nitrate_con', 'month']].resample('ME').mean().reset_index()
    df_monthly = df_monthly.dropna()
    return df_monthly

# Train on WQS0083
X_train = df_xgb[feature_cols_xgb + ['month']].values
y_train = df_xgb['nitrate_con'].values

model_xgb_final = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
model_xgb_final.fit(X_train, y_train)

# Test on each site
fig, axes = plt.subplots(len(test_sites), 1, figsize=(12, 4 * len(test_sites)))

for i, site in enumerate(test_sites):
    df_site = build_features(site)
    X_test = df_site[feature_cols_xgb + ['month']].values
    y_test = df_site['nitrate_con'].values
    preds = model_xgb_final.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    print(f"{site}: MSE={mse:.4f}")

    axes[i].plot(df_site['date'], y_test, label='actual')
    axes[i].plot(df_site['date'], preds, label='forecast')
    axes[i].xaxis.set_major_locator(mdates.YearLocator())
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_title(f'{site} - XGBoost (MSE={mse:.4f})')
    axes[i].legend()

plt.suptitle('XGBoost: Trained on WQS0083, Tested on Other Sites', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from data import get_site_ids
all_sites = get_site_ids()
test_sites_all = [s for s in all_sites if s != 'WQS0083']

results = {}
failed = []

for site in test_sites_all:
    try:
        df_site = build_features(site)
        if len(df_site) < 12:
            print(f"{site}: skipped (too few months)")
            continue
        X_test = df_site[feature_cols_xgb + ['month']].values
        y_test = df_site['nitrate_con'].values
        preds = model_xgb_final.predict(X_test)
        mse = mean_squared_error(y_test, preds)
        results[site] = {'mse': mse, 'df': df_site, 'preds': preds, 'y_test': y_test}
        print(f"{site}: MSE={mse:.4f}")
    except Exception as e:
        print(f"{site}: FAILED — {e}")
        failed.append(site)

print(f"\nSucceeded: {len(results)}  Failed: {len(failed)}")
print(f"Mean MSE across sites: {np.mean([v['mse'] for v in results.values()]):.4f}")
print(f"Std  MSE across sites: {np.std([v['mse'] for v in results.values()]):.4f}")

In [ ]:
n = len(results)
n_cols = 5
n_rows = (n + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, (site, res) in enumerate(results.items()):
    ax = axes[i]
    ax.plot(res['df']['date'], res['y_test'], label='actual')
    ax.plot(res['df']['date'], res['preds'], label='forecast')
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(f'{site}\nMSE={res["mse"]:.4f}')
    ax.legend(fontsize=7)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('XGBoost: Trained on WQS0083, Tested on All Sites', fontsize=14)
plt.tight_layout()
plt.show()